In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

# 1. TẠO DỮ LIỆU THÔ (Mô phỏng 5000 đơn hàng từ sàn TMĐT)
np.random.seed(42)
n_rows = 5000

categories = ['Điện thoại & Phụ kiện', 'Thời trang', 'Mỹ phẩm', 'Nhà cửa & Đời sống']
regions = ['Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Cần Thơ', 'Hải Phòng']

data = {
    'Order_ID': [f"ORD{str(i).zfill(5)}" for i in range(1, n_rows + 1)],
    'Order_Date': [datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(n_rows)],
    'Category': np.random.choice(categories, n_rows, p=[0.4, 0.3, 0.2, 0.1]),
    'Region': np.random.choice(regions, n_rows),
    'Gross_Amount': np.random.randint(10, 500, n_rows) * 10000, # Từ 100k đến 5 triệu
    'Delivery_Days': np.random.randint(1, 14, n_rows), # Thời gian giao hàng từ 1-14 ngày
    'Status': np.random.choice(['Delivered', 'Canceled', 'Returned'], n_rows, p=[0.75, 0.15, 0.10]),
    'Customer_Rating': np.random.randint(1, 6, n_rows) # Đánh giá 1-5 sao
}

df_raw = pd.DataFrame(data)

# Cố tình tạo ra "rác" để xử lý trong SQL
df_raw.loc[10:50, 'Gross_Amount'] = np.nan # Lỗi hệ thống không ghi nhận giá tiền
df_raw.loc[100:150, 'Status'] = 'Deliverd' # Lỗi chính tả do nhân viên nhập tay
df_raw.loc[df_raw['Status'] != 'Delivered', 'Customer_Rating'] = np.nan # Đơn hủy thì không có rating

# 2. ĐẨY VÀO SQLITE DATABASE
conn = sqlite3.connect('ecommerce_db.sqlite')
df_raw.to_sql('raw_orders', conn, if_exists='replace', index=False)

print("Đã tạo Database 'ecommerce_db.sqlite' và lưu bảng dữ liệu thô 'raw_orders' thành công!")
conn.close()

Đã tạo Database 'ecommerce_db.sqlite' và lưu bảng dữ liệu thô 'raw_orders' thành công!


In [2]:
# Kết nối lại Database
conn = sqlite3.connect('ecommerce_db.sqlite')
cursor = conn.cursor()

# Viết truy vấn SQL làm sạch dữ liệu
sql_clean = """
CREATE TABLE IF NOT EXISTS clean_orders AS
SELECT
    Order_ID,
    DATE(Order_Date) as Order_Date,
    STRFTIME('%Y-%m', Order_Date) as Order_Month,
    Category,
    Region,

    -- Xử lý tiền: Nếu rỗng (Null) hoặc âm thì gán bằng 0
    CASE
        WHEN Gross_Amount IS NULL OR Gross_Amount < 0 THEN 0
        ELSE Gross_Amount
    END as Gross_Amount,

    -- Sửa lỗi chính tả trạng thái đơn hàng
    CASE
        WHEN Status = 'Deliverd' THEN 'Delivered'
        ELSE Status
    END as Final_Status,

    Delivery_Days,

    -- Xử lý Rating: Nếu rỗng thì để là 0 (hoặc có thể bỏ qua khi tính trung bình)
    COALESCE(Customer_Rating, 0) as Customer_Rating,

    -- Cột tính Doanh thu thực tế (Chỉ lấy tiền của đơn Giao thành công)
    CASE
        WHEN Status = 'Delivered' THEN Gross_Amount
        ELSE 0
    END as Net_Revenue

FROM raw_orders;
"""

cursor.execute("DROP TABLE IF EXISTS clean_orders")
cursor.execute(sql_clean)
conn.commit()

# Tải bảng đã dọn dẹp ra file CSV để đưa lên Power BI
df_clean = pd.read_sql_query("SELECT * FROM clean_orders", conn)
df_clean.to_csv('Cleaned_Ecommerce_Data.csv', index=False)

print("Đã làm sạch bằng SQL và xuất file 'Cleaned_Ecommerce_Data.csv' cho Power BI!")
conn.close()

Đã làm sạch bằng SQL và xuất file 'Cleaned_Ecommerce_Data.csv' cho Power BI!
